# Answer-set programming — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def truth_rows(names):
    return [dict(zip(names, vals)) for vals in itertools.product([False, True], repeat=len(names))]
def draw_graph(nodes, edges, title='graph'):
    angles=np.linspace(0, 2*np.pi, len(nodes), endpoint=False)
    pos={n:(np.cos(a), np.sin(a)) for n,a in zip(nodes, angles)}
    plt.figure(figsize=(5,4))
    for a,b in edges:
        xa,ya=pos[a]; xb,yb=pos[b]; plt.plot([xa,xb],[ya,yb], 'k-', alpha=.6)
    for n,(x,y) in pos.items():
        plt.scatter([x],[y], s=500, zorder=3); plt.text(x,y,n,ha='center',va='center',color='white',weight='bold')
    plt.axis('equal'); plt.axis('off'); plt.title(title)
    return pos
print('setup ok')

## Stable models as self-consistent worlds
Answer-set programming searches for sets of atoms that justify themselves under rules with negation-as-failure. The examples show a choice rule, a constraint, stable-model filtering, optimization, and a tiny scheduling world.

In [ ]:
# 1) choice by negation-as-failure: exactly one of a or b
models=[set(m) for r in range(3) for m in itertools.combinations(['a','b'],r)]; stable=[m for m in models if (('a' in m) ^ ('b' in m))]
plt.figure(figsize=(5,3)); plt.bar(['{}','{a}','{b}','{a,b}'], [m in stable for m in models], color='tab:green'); plt.title('stable choices')
assert stable==[{'a'},{'b'}]

In [ ]:
# 2) constraint removes worlds containing both selected atoms
worlds=[set(m) for r in range(3) for m in itertools.combinations(['takeA','takeB'],r)]; ok=[not ({'takeA','takeB'} <= w) for w in worlds]
plt.figure(figsize=(5,3)); plt.bar(range(4), ok, color=['tab:green' if x else 'tab:red' for x in ok]); plt.xticks(range(4), [str(w) for w in worlds], rotation=20); plt.title('constraint: not both')
assert ok.count(False)==1

In [ ]:
# 3) support: atoms without supporting rules are rejected
rules={('light',):'see'}; candidates=[set(),{'see'},{'light'},{'light','see'}]
valid=[('see' not in c) or ('light' in c) for c in candidates]
plt.figure(figsize=(6,3)); plt.bar(range(4), valid, color='tab:purple'); plt.xticks(range(4), [str(c) for c in candidates], rotation=20); plt.title('unsupported see is invalid')
assert valid==[True, False, True, True]

In [ ]:
# 4) optimization chooses the minimum-cost stable model
stable=[{'a'},{'b'}]; cost=lambda m: 2 if 'a' in m else 1; costs=[cost(m) for m in stable]
plt.figure(figsize=(5,3)); plt.bar(['{a}','{b}'], costs, color=['tab:red','tab:green']); plt.title('minimize cost')
assert stable[int(np.argmin(costs))]=={'b'}

In [ ]:
# 5) tiny schedule: one task in one slot, no conflict
assign=[('T1','AM'),('T1','PM')]; models=[{a} for a in assign]; ok=[True,True]
plt.figure(figsize=(5,3)); plt.bar([f'{t}-{s}' for (t,s), in models], ok, color='tab:blue'); plt.title('two answer sets: AM or PM')
assert len(models)==2 and all(len(m)==1 for m in models)